In [85]:
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import numpy as np
import semopy
import json
from scipy import stats
from sklearn.preprocessing import StandardScaler

In [116]:
colunas_interesse = ["Modelo", "chi2 p-value", "RMSEA", "AIC", "BIC"]
df_modelos_controle = pd.read_csv(r"..\resultados\resultados_empiricos\modelos_controle_prep.csv", index_col="Modelo", usecols=colunas_interesse)
df_modelos_autismo = pd.read_csv(r"..\resultados\resultados_empiricos\modelos_autismo_prep.csv", index_col="Modelo", usecols=colunas_interesse)

df_controle_prep = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_controle_prep.csv")
df_autismo_prep = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo_prep.csv")

df_controle = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_controle.csv")
df_autismo = pd.read_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo.csv")


# scaler = StandardScaler()
# df_controle[df_controle.columns] = scaler.fit_transform(df_controle)
# df_autismo[df_autismo.columns] = scaler.fit_transform(df_autismo)

with open(r"..\utils\fm_modelos_5_conexo.json", "r") as file:
    formula_modelos_conexos = json.load(file)

### 1) Filtragem de modelos com base em RMSEA < 0.08

In [87]:
df_controle_filtro1 = df_modelos_controle[df_modelos_controle["RMSEA"] < 0.08]
df_autismo_filtro1 = df_modelos_autismo[df_modelos_autismo["RMSEA"] < 0.08]

print(df_controle_filtro1.shape)
print(df_autismo_filtro1.shape)

(375, 4)
(359, 4)


### 2) Selecionando modelos que passaram na etapa 1, simultaneamente para ambos os conjuntos

In [88]:
modelos_filtro2 = pd.merge(df_controle_filtro1, 
                           df_autismo_filtro1, 
                           how="inner", 
                           left_index=True, 
                           right_index=True, 
                           suffixes=["_controle", "_autismo"])
modelos_filtro2.shape

(255, 8)

### 3) Verificar os TOP 10 modelos com base em ordenação de AIC/BIC

#### 3.1) Ordenação com base no AIC/BIC produzido pelo Conjunto Controle:

In [89]:
modelos_filtro2.sort_values("BIC_controle").head(10)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M08682_Edges4,0.001055,0.071795,17.915579,56.354475,0.031758,0.067727,17.903044,50.775447
M08812_Edges4,0.001055,0.071795,17.915579,56.354475,0.031759,0.067727,17.903045,50.775447
M03528_Edges4,1.000000,0.000000,17.999997,56.438893,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,18.000000,56.438895,1.000000,0.000000,18.000000,50.872403
M00001_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872402
M25888_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872403
M10302_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872403
M09102_Edges5,0.000552,0.079957,19.917286,62.627170,0.098826,0.054826,19.934959,56.459850
M08640_Edges5,0.000552,0.079957,19.917286,62.627170,0.098829,0.054826,19.934959,56.459851


Percebe-se que o Top 5 tem uma discrepância em BIC relevante a partir do 6º modelo

In [90]:
modelos_filtro2.sort_values("BIC_controle").head(5)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M08682_Edges4,0.001055,0.071795,17.915579,56.354475,0.031758,0.067727,17.903044,50.775447
M08812_Edges4,0.001055,0.071795,17.915579,56.354475,0.031759,0.067727,17.903045,50.775447
M03528_Edges4,1.000000,0.000000,17.999997,56.438893,1.000000,0.000000,18.000000,50.872403
M00934_Edges4,1.000000,0.000000,18.000000,56.438895,1.000000,0.000000,18.000000,50.872403
M00001_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872402


#### 3.2) Ordenação com base no AIC/BIC produzido pelo Conjunto Autismo:

In [91]:
modelos_filtro2.sort_values("BIC_autismo").head(10)

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M08682_Edges4,0.001055,0.071795,17.915579,56.354475,0.031758,0.067727,17.903044,50.775447
M08812_Edges4,0.001055,0.071795,17.915579,56.354475,0.031759,0.067727,17.903045,50.775447
M00001_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872402
M00934_Edges4,1.000000,0.000000,18.000000,56.438895,1.000000,0.000000,18.000000,50.872403
M03528_Edges4,1.000000,0.000000,17.999997,56.438893,1.000000,0.000000,18.000000,50.872403
M10302_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872403
M25888_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872403
M25373_Edges5,0.250205,0.024796,19.974960,62.684844,0.080281,0.058305,19.931037,56.455929
M25243_Edges5,0.250209,0.024796,19.974960,62.684844,0.080282,0.058305,19.931037,56.455929


Percebe-se que o Top 5 tem uma discrepância em BIC relevante a partir do 6º modelo

In [92]:
top_5 = modelos_filtro2.sort_values("BIC_autismo").head(7)
top_5_nomes = top_5.index.values
top_5 

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M08682_Edges4,0.001055,0.071795,17.915579,56.354475,0.031758,0.067727,17.903044,50.775447
M08812_Edges4,0.001055,0.071795,17.915579,56.354475,0.031759,0.067727,17.903045,50.775447
M00001_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,17.999999,50.872402
M00934_Edges4,1.000000,0.000000,18.000000,56.438895,1.000000,0.000000,18.000000,50.872403
M03528_Edges4,1.000000,0.000000,17.999997,56.438893,1.000000,0.000000,18.000000,50.872403
M10302_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872403
M25888_Edges4,1.000000,0.000000,18.000000,56.438896,1.000000,0.000000,18.000000,50.872403


In [121]:
formula_modelos_conexos["M08682_Edges4"]

{'formula': 'Visual ~ Default_mode + Cerebellar\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~ Somatomotor + Control\nControl ~~ Control',
 'Arestas': ['Somatomotor->Cerebellar',
  'Default_mode->Visual',
  'Cerebellar->Visual',
  'Control->Cerebellar'],
 'Num_Arestas': 4}

In [122]:
formula_modelos_conexos["M08812_Edges4"]

{'formula': 'Visual ~ Default_mode + Cerebellar\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Somatomotor + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control',
 'Arestas': ['Somatomotor->Default_mode',
  'Default_mode->Visual',
  'Cerebellar->Visual',
  'Control->Default_mode'],
 'Num_Arestas': 4}

### 4) Filtragem por chi quadrado > 0.05

In [93]:
top_filtro_chi = top_5[(top_5["chi2 p-value_autismo"] > 0.05)&(top_5["chi2 p-value_controle"] > 0.05)]
top_filtro_chi

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M00001_Edges4,1.0,0.0,18.000000,56.438896,1.0,0.0,17.999999,50.872402
M00934_Edges4,1.0,0.0,18.000000,56.438895,1.0,0.0,18.000000,50.872403
M03528_Edges4,1.0,0.0,17.999997,56.438893,1.0,0.0,18.000000,50.872403
M10302_Edges4,1.0,0.0,18.000000,56.438896,1.0,0.0,18.000000,50.872403
M25888_Edges4,1.0,0.0,18.000000,56.438896,1.0,0.0,18.000000,50.872403


### 5) Verificação de arestas estatisticamente significativas

In [94]:
def roda_os_modelos(df, modelos, ajuste_p=None, retornar_inspection=False):    
    df_resultados = pd.DataFrame()
    dicionario = dict()

    for nome, modelo in modelos.items():
        try:
            formula = modelo["formula"]
        except:
            formula = modelo
        
        model = semopy.Model(formula)
        model.fit(df, obj="MLW")
        stats = semopy.stats.calc_stats(model)
        df_resultados = pd.concat([df_resultados, stats])
        if retornar_inspection:
            edges_inspection = model.inspect()
            
            if ajuste_p:
                edges_inspection = edges_inspection[edges_inspection["op"] == "~"]
                p_valores = edges_inspection["p-value"]
                rejeitados, p_corrigidos_fdr, _, _ = multipletests(p_valores, alpha=0.05, method=ajuste_p)
                edges_inspection["p-value-adj"] = p_corrigidos_fdr
                edges_inspection["tipo_correcao"] = ajuste_p
                edges_inspection["rejeita"] = rejeitados
            dicionario[nome] = edges_inspection
            
   
    df_resultados["Modelo"] = modelos.keys()
    df_resultados.set_index("Modelo", inplace=True)

    if retornar_inspection:
        return df_resultados, dicionario

    return df_resultados

In [95]:
modelos_interesse = {m:v for m,v in formula_modelos_conexos.items() if m in top_filtro_chi.index}

#### 4.1) Arestas indicadas pelo Conjunto de Controle

In [129]:
_, dic_arestas_controle = roda_os_modelos(df_controle_prep, 
                                        modelos_interesse,
                                        ajuste_p="bonferroni", 
                                        retornar_inspection=True)
for m in modelos_interesse.keys():
    print(m)
    display(dic_arestas_controle[m])

M00001_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Control,~,Visual,0.101172,0.040732,2.483864,1.299654e-02,5.198616e-02,bonferroni,False
1,Control,~,Somatomotor,0.212962,0.041278,5.159241,2.479526e-07,9.918103e-07,bonferroni,True
2,Control,~,Default_mode,0.333623,0.039591,8.426640,0.000000e+00,0.000000e+00,bonferroni,True
3,Control,~,Cerebellar,0.156906,0.039617,3.960596,7.476302e-05,2.990521e-04,bonferroni,True


M00934_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Cerebellar,~,Visual,0.261108,0.042843,6.094500,1.097802e-09,4.391207e-09,bonferroni,True
1,Cerebellar,~,Somatomotor,0.232740,0.044625,5.215502,1.833205e-07,7.332819e-07,bonferroni,True
2,Cerebellar,~,Default_mode,-0.080830,0.045473,-1.777530,7.548114e-02,3.019245e-01,bonferroni,False
3,Cerebellar,~,Control,0.183324,0.046351,3.955085,7.650735e-05,3.060294e-04,bonferroni,True


M03528_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Default_mode,~,Visual,0.186054,0.041445,4.489131,0.000007,0.000029,bonferroni,True
1,Default_mode,~,Somatomotor,0.200166,0.042735,4.683920,0.000003,0.000011,bonferroni,True
2,Default_mode,~,Cerebellar,-0.073336,0.041316,-1.774987,0.075900,0.303600,bonferroni,False
3,Default_mode,~,Control,0.355237,0.042087,8.440501,0.000000,0.000000,bonferroni,True


M10302_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Somatomotor,~,Visual,0.137922,0.041685,3.308704,9.372887e-04,3.749155e-03,bonferroni,True
1,Somatomotor,~,Default_mode,0.198418,0.042473,4.671599,2.988633e-06,1.195453e-05,bonferroni,True
2,Somatomotor,~,Cerebellar,0.210254,0.040297,5.217574,1.812823e-07,7.251292e-07,bonferroni,True
3,Somatomotor,~,Control,0.224701,0.043611,5.152348,2.572444e-07,1.028978e-06,bonferroni,True


M25888_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Visual,~,Somatomotor,0.147016,0.044437,3.308391,9.383359e-04,3.753343e-03,bonferroni,True
1,Visual,~,Default_mode,0.197659,0.043912,4.501230,6.756149e-06,2.702460e-05,bonferroni,True
2,Visual,~,Cerebellar,0.250934,0.041244,6.084193,1.170796e-09,4.683184e-09,bonferroni,True
3,Visual,~,Control,0.114442,0.045876,2.494590,1.261028e-02,5.044112e-02,bonferroni,False


#### 4.2) Arestas indicadas pelo Conjunto de Autismo

In [130]:
_, dic_arestas_autismo = roda_os_modelos(df_autismo_prep, 
                                        modelos_interesse,
                                        ajuste_p="bonferroni", 
                                        retornar_inspection=True)
for m in modelos_interesse.keys():
    print(m)
    display(dic_arestas_autismo[m])

M00001_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Control,~,Visual,0.115832,0.050322,2.301828,2.134488e-02,8.537952e-02,bonferroni,False
1,Control,~,Somatomotor,0.210378,0.055829,3.768237,1.644047e-04,6.576186e-04,bonferroni,True
2,Control,~,Default_mode,0.420271,0.053312,7.883272,3.108624e-15,1.243450e-14,bonferroni,True
3,Control,~,Cerebellar,0.168629,0.048525,3.475114,5.106366e-04,2.042546e-03,bonferroni,True


M00934_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Cerebellar,~,Visual,0.211197,0.059417,3.554457,3.787608e-04,1.515043e-03,bonferroni,True
1,Cerebellar,~,Somatomotor,0.383588,0.064499,5.947149,2.728532e-09,1.091413e-08,bonferroni,True
2,Cerebellar,~,Default_mode,-0.204911,0.069284,-2.957561,3.100831e-03,1.240332e-02,bonferroni,True
3,Cerebellar,~,Control,0.241052,0.069372,3.474790,5.112535e-04,2.045014e-03,bonferroni,True


M03528_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Default_mode,~,Visual,0.224948,0.049368,4.556595,5.198941e-06,2.079576e-05,bonferroni,True
1,Default_mode,~,Somatomotor,0.295932,0.054856,5.394741,6.862254e-08,2.744902e-07,bonferroni,True
2,Default_mode,~,Cerebellar,-0.145243,0.049130,-2.956300,3.113543e-03,1.245417e-02,bonferroni,True
3,Default_mode,~,Control,0.426048,0.054045,7.883197,3.108624e-15,1.243450e-14,bonferroni,True


M10302_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Somatomotor,~,Visual,0.074998,0.052409,1.431023,1.524236e-01,6.096943e-01,bonferroni,False
1,Somatomotor,~,Default_mode,0.313006,0.058039,5.392997,6.929194e-08,2.771677e-07,bonferroni,True
2,Somatomotor,~,Cerebellar,0.287730,0.048392,5.945848,2.750294e-09,1.100117e-08,bonferroni,True
3,Somatomotor,~,Control,0.225384,0.059882,3.763784,1.673617e-04,6.694468e-04,bonferroni,True


M25888_Edges4


,lval,op,rval,Estimate,Std. Err,z-value,p-value,p-value-adj,tipo_correcao,rejeita
0,Visual,~,Somatomotor,0.095195,0.066470,1.432152,0.152100,0.608401,bonferroni,False
1,Visual,~,Default_mode,0.301844,0.066246,4.556377,0.000005,0.000021,bonferroni,True
2,Visual,~,Cerebellar,0.200937,0.056539,3.553946,0.000379,0.001518,bonferroni,True
3,Visual,~,Control,0.158055,0.068456,2.308853,0.020952,0.083807,bonferroni,False


### 6) Poda de modelos e comparação com modelos não podados

#### 6.1) Poda de arestas sem Bonferroni

In [115]:
modelos_podados_bonf = {
    "M00934_Edges4_poda_CeDM": 'Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~ Visual + Somatomotor + 0*Default_mode + Control\nControl ~~ Control',
    "M03528_Edges4_poda_DMCe":'Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Visual + Somatomotor + 0*Cerebellar + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control',
    "M00001_Edges4_poda_CV": "Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~ 0*Visual + Somatomotor + Default_mode + Cerebellar",
    "M10302_Edges4_poda_SV": "Visual ~~ Visual\nSomatomotor ~ 0*Visual + Default_mode + Cerebellar + Control\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_poda_VC": 'Visual ~ Somatomotor + Default_mode + Cerebellar + 0*Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control',
    "M25888_Edges4_poda_VS_VC": "Visual ~ 0*Somatomotor + Default_mode + Cerebellar + 0*Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control"
}

modelos_podados_sem_bonf = {
    "M03528_Edges4_poda_DMCe":'Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~ Visual + Somatomotor + 0*Cerebellar + Control\nCerebellar ~~ Cerebellar\nControl ~~ Control',
    "M00934_Edges4_poda_CeDM": 'Visual ~~ Visual\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~ Visual + Somatomotor + 0*Default_mode + Control\nControl ~~ Control',
    "M10302_Edges4_poda_SV": "Visual ~~ Visual\nSomatomotor ~ 0*Visual + Default_mode + Cerebellar + Control\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
    "M25888_Edges4_poda_VS": "Visual ~ 0*Somatomotor + Default_mode + Cerebellar + Control\nSomatomotor ~~ Somatomotor\nDefault_mode ~~ Default_mode\nCerebellar ~~ Cerebellar\nControl ~~ Control",
}

In [117]:
res_autismo, dic_arestas_autismo = roda_os_modelos(
                                        df_autismo_prep, 
                                        modelos_podados_sem_bonf,
                                        retornar_inspection=True)
res_autismo = res_autismo.reset_index()[colunas_interesse].set_index("Modelo")


res_controle, dic_arestas_controle = roda_os_modelos(
                                        df_controle_prep, 
                                        modelos_podados_sem_bonf,
                                        retornar_inspection=True)
res_controle = res_controle.reset_index()[colunas_interesse].set_index("Modelo")



res_podados = pd.merge(res_controle, 
         res_autismo, 
         how="inner", 
         left_index=True, 
         right_index=True, 
         suffixes=["_controle", "_autismo"])


display(res_podados)

for m in modelos_podados_sem_bonf.keys():
    print(f"\n{m}:")
    print("CONTROLE")
    display(dic_arestas_controle[m])

    print("\nAUTISMO")
    display(dic_arestas_autismo[m])

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M03528_Edges4_poda_DMCe,0.872020,0.000000,15.988139,50.156046,0.281594,0.028491,15.939553,45.159467
M00934_Edges4_poda_CeDM,0.872020,0.000000,15.988139,50.156046,0.281595,0.028491,15.939553,45.159467
M10302_Edges4_poda_SV,0.146100,0.032201,15.959046,50.126953,0.957385,0.000000,15.985659,45.205572
M25888_Edges4_poda_VS,0.146098,0.032201,15.959046,50.126953,0.957385,0.000000,15.985659,45.205572



M03528_Edges4_poda_DMCe:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Default_mode,~,Visual,0.168610,0.040301,4.183804,0.000029
1,Default_mode,~,Somatomotor,0.183836,0.041912,4.386243,0.000012
2,Default_mode,~,Cerebellar,0.000000,-,-,-
3,Default_mode,~,Control,0.343209,0.041729,8.224666,0.0
4,Visual,~~,Visual,1.000000,0.047144,21.211512,0.0
5,Somatomotor,~~,Somatomotor,1.000000,0.044486,22.478791,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.047441,21.07883,0.0
7,Control,~~,Control,1.000000,0.045835,21.81727,0.0
8,Default_mode,~~,Default_mode,0.699289,0.042998,16.263456,0.0



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Default_mode,~,Visual,0.200356,0.049395,4.056216,0.00005
1,Default_mode,~,Somatomotor,0.247415,0.053159,4.654283,0.000003
2,Default_mode,~,Cerebellar,0.000000,-,-,-
3,Default_mode,~,Control,0.402979,0.054294,7.422185,0.0
4,Visual,~~,Visual,1.000000,0.057694,17.33276,0.0
5,Somatomotor,~~,Somatomotor,1.000000,0.047359,21.11546,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.058293,17.154709,0.0
7,Control,~~,Control,1.000000,0.048736,20.518568,0.0
8,Default_mode,~~,Default_mode,0.491248,0.041152,11.937336,0.0


...

,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Somatomotor,~,Visual,0.000000,-,-,-
1,Somatomotor,~,Default_mode,0.338159,0.055529,6.089789,0.0
2,Somatomotor,~,Cerebellar,0.304860,0.04703,6.482203,0.0
3,Somatomotor,~,Control,0.239095,0.059334,4.029638,0.000056
4,Visual,~~,Visual,1.000000,0.054296,18.417433,0.0
5,Default_mode,~~,Default_mode,1.000000,0.045986,21.745947,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.063682,15.702924,0.0
7,Control,~~,Control,1.000000,0.043198,23.14942,0.0
8,Somatomotor,~~,Somatomotor,0.507824,0.042541,11.937336,0.0



M25888_Edges4_poda_VS:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.231637,0.043143,5.368992,0.0
2,Visual,~,Cerebellar,0.287832,0.04013,7.172468,0.0
3,Visual,~,Control,0.150542,0.045019,3.343946,0.000826
4,Somatomotor,~~,Somatomotor,1.000000,0.043569,22.951885,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044877,22.283314,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.050487,19.807011,0.0
7,Control,~~,Control,1.000000,0.041151,24.300592,0.0
8,Visual,~~,Visual,0.751627,0.046216,16.263456,0.0



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.334177,0.062547,5.342852,0.0
2,Visual,~,Cerebellar,0.229869,0.052974,4.339286,0.000014
3,Visual,~,Control,0.180521,0.066833,2.701085,0.006911
4,Somatomotor,~~,Somatomotor,1.000000,0.043198,23.149431,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044263,22.591988,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.059378,16.841265,0.0
7,Control,~~,Control,1.000000,0.041287,24.220457,0.0
8,Visual,~~,Visual,0.644291,0.053973,11.937336,0.0


#### 6.2) Poda com Bonferroni

In [118]:
res_autismo, dic_arestas_autismo = roda_os_modelos(
                                        df_autismo_prep, 
                                        modelos_podados_bonf,
                                        retornar_inspection=True)
res_autismo = res_autismo.reset_index()[colunas_interesse].set_index("Modelo")


res_controle, dic_arestas_controle = roda_os_modelos(
                                        df_controle_prep, 
                                        modelos_podados_bonf,
                                        retornar_inspection=True)
res_controle = res_controle.reset_index()[colunas_interesse].set_index("Modelo")



res_podados = pd.merge(res_controle, 
         res_autismo, 
         how="inner", 
         left_index=True, 
         right_index=True, 
         suffixes=["_controle", "_autismo"])


display(res_podados)

for m in modelos_podados_bonf.keys():
    print(f"\n{m}:")
    print("CONTROLE")
    display(dic_arestas_controle[m])

    print("\nAUTISMO")
    display(dic_arestas_autismo[m])

,chi2 p-value_controle,RMSEA_controle,AIC_controle,BIC_controle,chi2 p-value_autismo,RMSEA_autismo,AIC_autismo,BIC_autismo
Modelo,,,,,,,,
M00934_Edges4_poda_CeDM,0.872020,0.000000,15.988139,50.156046,0.281595,0.028491,15.939553,45.159467
M03528_Edges4_poda_DMCe,0.872020,0.000000,15.988139,50.156046,0.281594,0.028491,15.939553,45.159467
M00001_Edges4_poda_CV,0.519881,0.000000,15.976668,50.144575,0.625919,0.000000,15.962952,45.182866
M10302_Edges4_poda_SV,0.146100,0.032201,15.959046,50.126953,0.957385,0.000000,15.985659,45.205572
M25888_Edges4_poda_VC,0.519872,0.000000,15.976667,50.144575,0.625918,0.000000,15.962952,45.182866
M25888_Edges4_poda_VS_VC,0.005147,0.057319,13.917286,43.814205,0.320170,0.023627,13.934959,39.502384



M00934_Edges4_poda_CeDM:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Cerebellar,~,Visual,0.246933,0.042269,5.841895,0.0
1,Cerebellar,~,Somatomotor,0.217937,0.043959,4.957703,0.000001
2,Cerebellar,~,Default_mode,0.000000,-,-,-
3,Cerebellar,~,Control,0.155841,0.043768,3.560657,0.00037
4,Visual,~~,Visual,1.000000,0.04852,20.610244,0.0
5,Somatomotor,~~,Somatomotor,1.000000,0.04482,22.311603,0.0
6,Default_mode,~~,Default_mode,1.000000,0.043312,23.08834,0.0
7,Control,~~,Control,1.000000,0.041757,23.948244,0.0
8,Cerebellar,~~,Cerebellar,0.769273,0.047301,16.263456,0.0



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Cerebellar,~,Visual,0.170136,0.058662,2.900285,0.003728
1,Cerebellar,~,Somatomotor,0.332789,0.063132,5.271331,0.0
2,Cerebellar,~,Default_mode,0.000000,-,-,-
3,Cerebellar,~,Control,0.158600,0.06448,2.459675,0.013906
4,Visual,~~,Visual,1.000000,0.056071,17.834636,0.0
5,Somatomotor,~~,Somatomotor,1.000000,0.047788,20.925872,0.0
6,Default_mode,~~,Default_mode,1.000000,0.04182,23.912237,0.0
7,Control,~~,Control,1.000000,0.041761,23.945791,0.0
8,Cerebellar,~~,Cerebellar,0.692870,0.058042,11.937336,0.0


...

,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.130932,0.065261,2.006299,0.044824
1,Visual,~,Default_mode,0.375060,0.058689,6.390609,0.0
2,Visual,~,Cerebellar,0.231870,0.055445,4.18199,0.000029
3,Visual,~,Control,0.000000,-,-,-
4,Somatomotor,~~,Somatomotor,1.000000,0.043198,23.149431,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044263,22.591988,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.059378,16.841265,0.0
7,Control,~~,Control,1.000000,0.041287,24.220457,0.0
8,Visual,~~,Visual,0.651603,0.054585,11.937336,0.0



M25888_Edges4_poda_VS_VC:
CONTROLE


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.297029,0.038833,7.648893,0.0
2,Visual,~,Cerebellar,0.326398,0.038833,8.405191,0.0
3,Visual,~,Control,0.000000,-,-,-
4,Somatomotor,~~,Somatomotor,1.000000,0.043569,22.951885,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044877,22.283314,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.050487,19.807011,0.0
7,Control,~~,Control,1.000000,0.041151,24.300592,0.0
8,Visual,~~,Visual,0.767336,0.047182,16.263456,0.0



AUTISMO


,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,Visual,~,Somatomotor,0.000000,-,-,-
1,Visual,~,Default_mode,0.437084,0.050221,8.703171,0.0
2,Visual,~,Cerebellar,0.280427,0.050221,5.583835,0.0
3,Visual,~,Control,0.000000,-,-,-
4,Somatomotor,~~,Somatomotor,1.000000,0.043198,23.149431,0.0
5,Default_mode,~~,Default_mode,1.000000,0.044263,22.591988,0.0
6,Cerebellar,~~,Cerebellar,1.000000,0.059378,16.841265,0.0
7,Control,~~,Control,1.000000,0.041287,24.220457,0.0
8,Visual,~~,Visual,0.661166,0.055386,11.937336,0.0


### 7) Teste de diferença de parâmetros das arestas:

#### 7.1) Teste z

In [119]:
for nome, formula in modelos_interesse.items():

    # --- Grupo Controle ---
    model_controle = semopy.Model(formula["formula"])
    model_controle.fit(df_controle_prep)
    arestas_controle = model_controle.inspect()
    arestas_controle = arestas_controle[(arestas_controle["op"] == "~") & (arestas_controle["Std. Err"] != "-")].copy()

    # --- Grupo Autismo ---
    model_autismo = semopy.Model(formula["formula"])
    model_autismo.fit(df_autismo_prep)
    arestas_autismo = model_autismo.inspect()
    arestas_autismo = arestas_autismo[(arestas_autismo["op"] == "~") & (arestas_autismo["Std. Err"] != "-")].copy()

    df_comparacao = pd.merge(
        arestas_controle, 
        arestas_autismo, 
        on=["lval", "op", "rval"], 
        suffixes=('_ctrl', '_tea')
    )

    # Cálculo do Z e P-valor usando as colunas mescladas
    z_diff = (df_comparacao["Estimate_ctrl"] - df_comparacao["Estimate_tea"]) / np.sqrt((df_comparacao["Std. Err_ctrl"]**2) + (df_comparacao["Std. Err_tea"]**2))
    p_valor = 2 * (1 - norm.cdf(np.abs(z_diff)))
    
    # Guardando os resultados no dataframe
    df_comparacao["z_diff"] = z_diff
    df_comparacao["p_valor"] = p_valor
    
    print(nome)
    colunas = ["lval", "rval", "Estimate_ctrl", "p-value_ctrl", "Estimate_tea", "p-value_tea", "p_valor"]
    display(df_comparacao[colunas])

M00001_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Control,Visual,0.101172,1.299654e-02,0.115832,2.134488e-02,0.820850
1,Control,Somatomotor,0.212962,2.479526e-07,0.210378,1.644047e-04,0.970316
2,Control,Default_mode,0.333623,0.000000e+00,0.420271,3.108624e-15,0.191948
3,Control,Cerebellar,0.156906,7.476302e-05,0.168629,5.106366e-04,0.851562


M00934_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Cerebellar,Visual,0.261108,1.097802e-09,0.211197,3.787608e-04,0.495645
1,Cerebellar,Somatomotor,0.232740,1.833205e-07,0.383588,2.728532e-09,0.054441
2,Cerebellar,Default_mode,-0.080830,7.548114e-02,-0.204911,3.100831e-03,0.134331
3,Cerebellar,Control,0.183324,7.650735e-05,0.241052,5.112535e-04,0.488988


M03528_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Default_mode,Visual,0.186054,0.000007,0.224948,5.198941e-06,0.546248
1,Default_mode,Somatomotor,0.200166,0.000003,0.295932,6.862254e-08,0.168452
2,Default_mode,Cerebellar,-0.073336,0.075900,-0.145243,3.113543e-03,0.262644
3,Default_mode,Control,0.355237,0.000000,0.426048,3.108624e-15,0.301259


M10302_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Somatomotor,Visual,0.137922,9.372887e-04,0.074998,1.524236e-01,0.347395
1,Somatomotor,Default_mode,0.198418,2.988633e-06,0.313006,6.929194e-08,0.111100
2,Somatomotor,Cerebellar,0.210254,1.812823e-07,0.287730,2.750294e-09,0.218583
3,Somatomotor,Control,0.224701,2.572444e-07,0.225384,1.673617e-04,0.992647


M25888_Edges4


,lval,rval,Estimate_ctrl,p-value_ctrl,Estimate_tea,p-value_tea,p_valor
0,Visual,Somatomotor,0.147016,9.383359e-04,0.095195,0.152100,0.516901
1,Visual,Default_mode,0.197659,6.756149e-06,0.301844,0.000005,0.189907
2,Visual,Cerebellar,0.250934,1.170796e-09,0.200937,0.000379,0.474972
3,Visual,Control,0.114442,1.261028e-02,0.158055,0.020952,0.596643


In [120]:
# Z-score para 95% de confiança
Z_95 = 1.96 

for nome, formula in modelos_interesse.items():

    # --- Grupo Controle ---
    model_controle = semopy.Model(formula["formula"])
    model_controle.fit(df_controle_prep)
    arestas_controle = model_controle.inspect()
    arestas_controle = arestas_controle[(arestas_controle["op"] == "~") & (arestas_controle["Std. Err"] != "-")].copy()

    # Garantir que Std. Err é numérico (previne erros em algumas versões do semopy)
    arestas_controle["Std. Err"] = pd.to_numeric(arestas_controle["Std. Err"])

    # --- Grupo Autismo ---
    model_autismo = semopy.Model(formula["formula"])
    model_autismo.fit(df_autismo_prep)
    arestas_autismo = model_autismo.inspect()
    arestas_autismo = arestas_autismo[(arestas_autismo["op"] == "~") & (arestas_autismo["Std. Err"] != "-")].copy()
    
    # Garantir que Std. Err é numérico
    arestas_autismo["Std. Err"] = pd.to_numeric(arestas_autismo["Std. Err"])

    df_comparacao = pd.merge(
        arestas_controle, 
        arestas_autismo, 
        on=["lval", "op", "rval"], 
        suffixes=('_ctrl', '_tea')
    )

    # =====================================================================
    # NOVAS ADIÇÕES: CÁLCULO DOS INTERVALOS DE CONFIANÇA (95%)
    # =====================================================================
    
    # IC 95% Grupo Controle
    df_comparacao["IC_inf_ctrl"] = df_comparacao["Estimate_ctrl"] - (Z_95 * df_comparacao["Std. Err_ctrl"])
    df_comparacao["IC_sup_ctrl"] = df_comparacao["Estimate_ctrl"] + (Z_95 * df_comparacao["Std. Err_ctrl"])
    
    # IC 95% Grupo Autismo (TEA)
    df_comparacao["IC_inf_tea"] = df_comparacao["Estimate_tea"] - (Z_95 * df_comparacao["Std. Err_tea"])
    df_comparacao["IC_sup_tea"] = df_comparacao["Estimate_tea"] + (Z_95 * df_comparacao["Std. Err_tea"])
    
    # SE da Diferença (já usado no seu cálculo do Z)
    se_diff = np.sqrt((df_comparacao["Std. Err_ctrl"]**2) + (df_comparacao["Std. Err_tea"]**2)) 
    df_comparacao["diff"] = df_comparacao["Estimate_ctrl"] - df_comparacao["Estimate_tea"]
    
    # IC 95% da Diferença entre os grupos
    df_comparacao["IC_inf_diff"] = df_comparacao["diff"] - (Z_95 * se_diff)
    df_comparacao["IC_sup_diff"] = df_comparacao["diff"] + (Z_95 * se_diff)
    # =====================================================================

    # Cálculo do Z e P-valor usando as colunas mescladas (Seu código original)
    z_diff = df_comparacao["diff"] / se_diff
    p_valor = 2 * (1 - norm.cdf(np.abs(z_diff)))
    
    # Guardando os resultados no dataframe
    df_comparacao["z_diff"] = z_diff
    df_comparacao["p_valor_diff"] = p_valor # Renomeei levemente para deixar claro que é o p-valor da diferença
    
    print(f"\nResultados para o modelo: {nome}")
    
    # Atualizei a lista de colunas para exibir os novos limites
    colunas = [
        "lval", "rval", 
        "Estimate_ctrl", "IC_inf_ctrl", "IC_sup_ctrl", "p-value_ctrl", 
        "Estimate_tea", "IC_inf_tea", "IC_sup_tea", "p-value_tea", 
        "IC_inf_diff", "IC_sup_diff", "p_valor_diff"
    ]
    
    # Formatação opcional para não exibir muitas casas decimais
    display(df_comparacao[colunas].round(4))
    display(df_comparacao[["lval", "rval", "diff", "IC_inf_diff", "IC_sup_diff", "z_diff", "p_valor_diff"]])


Resultados para o modelo: M00001_Edges4


,lval,rval,Estimate_ctrl,IC_inf_ctrl,IC_sup_ctrl,p-value_ctrl,Estimate_tea,IC_inf_tea,IC_sup_tea,p-value_tea,IC_inf_diff,IC_sup_diff,p_valor_diff
0,Control,Visual,0.1012,0.0213,0.1810,0.0130,0.1158,0.0172,0.2145,0.0213,-0.1416,0.1122,0.8209
1,Control,Somatomotor,0.2130,0.1321,0.2939,0.0000,0.2104,0.1010,0.3198,0.0002,-0.1335,0.1387,0.9703
2,Control,Default_mode,0.3336,0.2560,0.4112,0.0000,0.4203,0.3158,0.5248,0.0000,-0.2168,0.0435,0.1919
3,Control,Cerebellar,0.1569,0.0793,0.2346,0.0001,0.1686,0.0735,0.2637,0.0005,-0.1345,0.1111,0.8516


,lval,rval,diff,IC_inf_diff,IC_sup_diff,z_diff,p_valor_diff
0,Control,Visual,-0.014661,-0.141552,0.112231,-0.226452,0.820850
1,Control,Somatomotor,0.002584,-0.133503,0.138670,0.037212,0.970316
2,Control,Default_mode,-0.086648,-0.216801,0.043506,-1.304838,0.191948
3,Control,Cerebellar,-0.011722,-0.134502,0.111058,-0.187126,0.851562



Resultados para o modelo: M00934_Edges4


,lval,rval,Estimate_ctrl,IC_inf_ctrl,IC_sup_ctrl,p-value_ctrl,Estimate_tea,IC_inf_tea,IC_sup_tea,p-value_tea,IC_inf_diff,IC_sup_diff,p_valor_diff
0,Cerebellar,Visual,0.2611,0.1771,0.3451,0.0000,0.2112,0.0947,0.3277,0.0004,-0.0937,0.1935,0.4956
1,Cerebellar,Somatomotor,0.2327,0.1453,0.3202,0.0000,0.3836,0.2572,0.5100,0.0000,-0.3046,0.0029,0.0544
2,Cerebellar,Default_mode,-0.0808,-0.1700,0.0083,0.0755,-0.2049,-0.3407,-0.0691,0.0031,-0.0384,0.2865,0.1343
3,Cerebellar,Control,0.1833,0.0925,0.2742,0.0001,0.2411,0.1051,0.3770,0.0005,-0.2213,0.1058,0.4890


...

,lval,rval,Estimate_ctrl,IC_inf_ctrl,IC_sup_ctrl,p-value_ctrl,Estimate_tea,IC_inf_tea,IC_sup_tea,p-value_tea,IC_inf_diff,IC_sup_diff,p_valor_diff
0,Somatomotor,Visual,0.1379,0.0562,0.2196,0.0009,0.0750,-0.0277,0.1777,0.1524,-0.0683,0.1942,0.3474
1,Somatomotor,Default_mode,0.1984,0.1152,0.2817,0.0000,0.3130,0.1992,0.4268,0.0000,-0.2556,0.0264,0.1111
2,Somatomotor,Cerebellar,0.2103,0.1313,0.2892,0.0000,0.2877,0.1929,0.3826,0.0000,-0.2009,0.0460,0.2186
3,Somatomotor,Control,0.2247,0.1392,0.3102,0.0000,0.2254,0.1080,0.3428,0.0002,-0.1459,0.1445,0.9926


,lval,rval,diff,IC_inf_diff,IC_sup_diff,z_diff,p_valor_diff
0,Somatomotor,Visual,0.062924,-0.068327,0.194175,0.939655,0.347395
1,Somatomotor,Default_mode,-0.114588,-0.255552,0.026376,-1.593266,0.111100
2,Somatomotor,Cerebellar,-0.077476,-0.200904,0.045951,-1.230305,0.218583
3,Somatomotor,Control,-0.000683,-0.145879,0.144514,-0.009215,0.992647



Resultados para o modelo: M25888_Edges4


,lval,rval,Estimate_ctrl,IC_inf_ctrl,IC_sup_ctrl,p-value_ctrl,Estimate_tea,IC_inf_tea,IC_sup_tea,p-value_tea,IC_inf_diff,IC_sup_diff,p_valor_diff
0,Visual,Somatomotor,0.1470,0.0599,0.2341,0.0009,0.0952,-0.0351,0.2255,0.1521,-0.1049,0.2085,0.5169
1,Visual,Default_mode,0.1977,0.1116,0.2837,0.0000,0.3018,0.1720,0.4317,0.0000,-0.2600,0.0516,0.1899
2,Visual,Cerebellar,0.2509,0.1701,0.3318,0.0000,0.2009,0.0901,0.3118,0.0004,-0.0872,0.1872,0.4750
3,Visual,Control,0.1144,0.0245,0.2044,0.0126,0.1581,0.0239,0.2922,0.0210,-0.2051,0.1179,0.5966


,lval,rval,diff,IC_inf_diff,IC_sup_diff,z_diff,p_valor_diff
0,Visual,Somatomotor,0.051822,-0.104891,0.208535,0.648130,0.516901
1,Visual,Default_mode,-0.104185,-0.259963,0.051593,-1.310853,0.189907
2,Visual,Cerebellar,0.049997,-0.087171,0.187165,0.714413,0.474972
3,Visual,Control,-0.043612,-0.205129,0.117905,-0.529234,0.596643


### 8) Teste de Invariância global (Teste de Chow)

Utilizando dados sem Pré-processamento

In [124]:
# Faz sentido retirar o intercepto?
modelos_estrela = {
    "M00001_Edges4": "Control ~ Visual + Somatomotor + Default_mode + Cerebellar",
    "M00934_Edges4": "Cerebellar ~ Visual + Somatomotor + Default_mode + Control",
    "M03528_Edges4": "Default_mode ~ Visual + Somatomotor + Cerebellar + Control",
    "M10302_Edges4": "Somatomotor ~ Visual + Default_mode + Cerebellar + Control",
    "M25888_Edges4": "Visual ~ Somatomotor + Default_mode + Cerebellar + Control",
    }


df_total = pd.concat([df_controle, df_autismo])
retira_va_indep = ""

for nome, formula in modelos_estrela.items():

    # 1. Ajustar os modelos via OLS (Mínimos Quadrados Ordinários)
    modelo_total = smf.ols(formula, data=df_total, drop_cols=retira_va_indep).fit()
    modelo_controle = smf.ols(formula, data=df_controle, drop_cols=retira_va_indep).fit()
    modelo_autismo = smf.ols(formula, data=df_autismo, drop_cols=retira_va_indep).fit()

    # 2. Extrair a Soma dos Quadrados dos Resíduos (SSR - a métrica de erro)
    ssr_total = modelo_total.ssr
    ssr_controle = modelo_controle.ssr
    ssr_autismo = modelo_autismo.ssr


    # 3. Definir os parâmetros para o teste
    k = 5 # 4 preditores + 1 intercepto
    N_controle = len(df_controle) # 529
    N_autismo = len(df_autismo)   # 285

    # 4. Calcular a Estatística F:
    numerador = (ssr_total - (ssr_controle + ssr_autismo)) / k
    denominador = (ssr_controle + ssr_autismo) / (N_controle + N_autismo - 2 * k)
    estatistica_f = numerador / denominador

    # 5. Calcular o P-valor Global:
    graus_liberdade_numerador = k # 5 variáveis
    graus_liberdade_denominador = N_controle + N_autismo - 2 * k
    p_valor = stats.f.sf(estatistica_f, graus_liberdade_numerador, graus_liberdade_denominador)

    print(f"\n\nTESTANDO O MODELO {nome}")
    print(f"\nEstatística F: {estatistica_f:.4f}")
    print(f"P-valor Global: {p_valor:.6e}")
    print("-" * 55)

    if p_valor < 0.05:
        print("VEREDITO: Quebra Estrutural Detectada (Diferença Sistêmica).")
        print("A topologia de predição da Default_mode difere estatisticamente entre Controle e Autismo.")
    else:
        print("VEREDITO: Invariância Confirmada.")
        print("Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.")



TESTANDO O MODELO M00001_Edges4

Estatística F: 0.9185
P-valor Global: 4.682239e-01
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.


TESTANDO O MODELO M00934_Edges4

Estatística F: 1.9363
P-valor Global: 8.605296e-02
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.


TESTANDO O MODELO M03528_Edges4

Estatística F: 1.8650
P-valor Global: 9.809512e-02
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-rede seja diferente.


TESTANDO O MODELO M10302_Edges4

Estatística F: 0.8359
P-valor Global: 5.243255e-01
-------------------------------------------------------
VEREDITO: Invariância Confirmada.
Não há evidência de que a arquitetura funcional dessa sub-r

### Regressão linear - teste de parâmetros

In [104]:
resultados_controle = {}
resultados_autismo = {}


for nome_modelo, formula in modelos_estrela.items():
    print(f"===========================================================")
    print(f"Avaliando Modelo: {nome_modelo}")
    print(f"Fórmula: {formula}")
    print(f"===========================================================")
    
    # smf.ols = Ordinary Least Squares (Regressão Linear Múltipla)
    # .fit() ajusta os dados aos parâmetros da regressão
    modelo_controle = smf.ols(formula=formula, data=df_controle).fit()
    modelo_autismo = smf.ols(formula=formula, data=df_autismo).fit()

    
    # Salvando o modelo na nossa lista de resultados para consultas futuras
    resultados_controle[nome_modelo] = modelo_controle
    resultados_autismo[nome_modelo] = modelo_autismo

    print("\nCONTROLE")
    print(modelo_controle.summary())
    print("\n")

    print("AUTISMO")
    print(modelo_autismo.summary())
    print("\n")



Avaliando Modelo: M00001_Edges4
Fórmula: Control ~ Visual + Somatomotor + Default_mode + Cerebellar

CONTROLE
                            OLS Regression Results                            
Dep. Variable:                Control   R-squared:                       0.542
Model:                            OLS   Adj. R-squared:                  0.538
Method:                 Least Squares   F-statistic:                     154.9
Date:                Tue, 28 Apr 2026   Prob (F-statistic):           2.26e-87
Time:                        08:25:54   Log-Likelihood:                -1382.5
No. Observations:                 529   AIC:                             2775.
Df Residuals:                     524   BIC:                             2796.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------

In [105]:
resultados_controle = {}
resultados_autismo = {}
resultados_diferenca = {} # Dicionário para salvar os testes de diferença

for nome_modelo, formula in modelos_estrela.items():
    print(f"===========================================================")
    print(f"Avaliando Modelo: {nome_modelo}")
    print(f"Fórmula: {formula}")
    print(f"===========================================================")
    
    # Ajustando os modelos
    modelo_controle = smf.ols(formula=formula, data=df_controle).fit()
    modelo_autismo = smf.ols(formula=formula, data=df_autismo).fit()
    
    # Salvando os resultados
    resultados_controle[nome_modelo] = modelo_controle
    resultados_autismo[nome_modelo] = modelo_autismo

    # ---------------------------------------------------------
    # TESTE DE DIFERENÇA DOS PARÂMETROS
    # ---------------------------------------------------------
    print("\n--- TESTE DE DIFERENÇA ENTRE OS GRUPOS (Controle vs Autismo) ---")
    
    diferencas_modelo = []
    
    # Iterar por cada variável preditora usada na fórmula (incluindo o Intercept)
    for parametro in modelo_controle.params.index:
        # Extraindo os Betas (Coeficientes)
        beta_c = modelo_controle.params[parametro]
        beta_a = modelo_autismo.params[parametro]
        
        # Extraindo os Erros Padrão (Standard Errors - bse)
        se_c = modelo_controle.bse[parametro]
        se_a = modelo_autismo.bse[parametro]
        
        # Calculando a Estatística Z
        z_stat = (beta_c - beta_a) / np.sqrt(se_c**2 + se_a**2)
        
        # Calculando o P-valor (Bicaudal)
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        
        # Marcador visual de significância
        sig = "*" if p_value < 0.05 else ""
        
        print(f"Parâmetro: {parametro:15} | Z-Score: {z_stat:>8.4f} | P-valor: {p_value:>8.4f} {sig}")
        
        # Guardando em uma lista caso queira transformar em DataFrame depois
        diferencas_modelo.append({
            'Variavel': parametro,
            'Beta_Controle': beta_c,
            'Beta_Autismo': beta_a,
            'Z_Stat': z_stat,
            'P_Value': p_value
        })
        
    resultados_diferenca[nome_modelo] = pd.DataFrame(diferencas_modelo)
    print("----------------------------------------------------------------\n")

Avaliando Modelo: M00001_Edges4
Fórmula: Control ~ Visual + Somatomotor + Default_mode + Cerebellar

--- TESTE DE DIFERENÇA ENTRE OS GRUPOS (Controle vs Autismo) ---
Parâmetro: Intercept       | Z-Score:   1.6893 | P-valor:   0.0912 
Parâmetro: Visual          | Z-Score:   0.1034 | P-valor:   0.9176 
Parâmetro: Somatomotor     | Z-Score:  -0.1525 | P-valor:   0.8788 
Parâmetro: Default_mode    | Z-Score:  -0.8535 | P-valor:   0.3934 
Parâmetro: Cerebellar      | Z-Score:  -0.3992 | P-valor:   0.6897 
----------------------------------------------------------------

Avaliando Modelo: M00934_Edges4
Fórmula: Cerebellar ~ Visual + Somatomotor + Default_mode + Control

--- TESTE DE DIFERENÇA ENTRE OS GRUPOS (Controle vs Autismo) ---
Parâmetro: Intercept       | Z-Score:   0.7571 | P-valor:   0.4490 
Parâmetro: Visual          | Z-Score:   0.2970 | P-valor:   0.7665 
Parâmetro: Somatomotor     | Z-Score:  -1.4515 | P-valor:   0.1466 
Parâmetro: Default_mode    | Z-Score:   1.4866 | P-valor: 